# Solutions · 12 · Polymer melts and solutions

Worked solutions to the exercises of [notebook 12](../notebooks/12_polymer_melts.ipynb). Try each exercise
yourself before reading its solution - the learning happens in the attempt. Solutions are one way to
answer each question; other correct approaches exist.

In [1]:
try:                      # on Google Colab (or anywhere engrheo is missing): install it from GitHub
    import engrheo
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ktwyw/engrheo"], check=True)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import optimize
from engrheo import datasets, polymers
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 90, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

import inspect
from IPython.display import Code

def show_source(obj):
    """Display the source code of a library function or class."""
    return Code(inspect.getsource(obj), language="python")

def heading(text):
    print(f"\n{text}\n" + "=" * len(text))

In [2]:
pe = pd.read_csv(datasets.path("polyethylene_eta0_vs_mw.csv"), comment="#")
M, eta0 = pe.Mw_g_mol.to_numpy(), pe.eta0_Pa_s.to_numpy()
res = optimize.least_squares(lambda p: np.log10(polymers.zero_shear_viscosity(M, 10**p[0], 10**p[1], p[2])) - np.log10(eta0), [4.0, 0.0, 3.0])
Mc, eta_c, a = 10**res.x[0], 10**res.x[1], res.x[2]

## Exercise 1

Predict the zero-shear viscosity of a 150 kg/mol grade with the fitted relation. How uncertain is the prediction if the exponent is uncertain by +/-0.1?

In [3]:
M_new = 150e3
for da in (0.0, -0.1, 0.1):
    print(f"exponent {a + da:.2f}: eta0(150 kg/mol) = {polymers.zero_shear_viscosity(M_new, Mc, eta_c, a + da):.3g} Pa s")

exponent 3.38: eta0(150 kg/mol) = 1.78e+05 Pa s
exponent 3.28: eta0(150 kg/mol) = 1.24e+05 Pa s
exponent 3.48: eta0(150 kg/mol) = 2.57e+05 Pa s


An uncertainty of 0.1 in the exponent changes the prediction by a factor $(M/M_c)^{0.1}$ - here almost 1.5 either
way, because 150 kg/mol is far above $M_c$. Power-law extrapolations over large ranges amplify small errors in
the exponent; interpolation within the measured grades is much safer.

## Exercise 2

A grade must process at the same viscosity as before, but at a melt temperature 20 K higher (Arrhenius, E_a = 30 kJ/mol for polyethylene). By how much may its molar mass increase?

In [4]:
Ea, T1 = 30e3, 190 + 273.15
T2 = T1 + 20
aT = np.exp(Ea / 8.314 * (1 / T2 - 1 / T1))
print(f"20 K hotter: viscosity x {aT:.3f}; allowed Mw increase for the same viscosity: x {aT ** (-1 / a):.3f}")

20 K hotter: viscosity x 0.724; allowed Mw increase for the same viscosity: x 1.100


Twenty kelvin lower the viscosity by about a quarter; because $\eta_0 \propto M^{3.4}$, only about 10 % more molar
mass uses that margin up. Temperature and molar mass trade off steeply against each other.

## Exercise 3

**Implement it yourself:** estimate the exponent above $M_c$ with an ordinary straight-line fit of $\log\eta_0$ against $\log M$ for the grades above 10 kg/mol, and compare with the piecewise fit.

In [5]:
m = M > 10e3
slope = np.polyfit(np.log10(M[m]), np.log10(eta0[m]), 1)[0]
print(f"straight line above 10 kg/mol: exponent {slope:.2f};  piecewise fit: {a:.2f}")

straight line above 10 kg/mol: exponent 3.41;  piecewise fit: 3.38


Both agree within the scatter. The piecewise fit uses all the data and also locates $M_c$; the straight line needs
you to choose the range - include points too close to $M_c$ and the slope is pulled down.